#### <b>preparation

In [2]:
import pandas as pd
import numpy as np
import glob
import os
import re
import io
import math
import openmeteo_requests
import requests_cache
import calendar

import requests
from retry_requests import retry
from datetime import datetime, timedelta, timezone

pd.set_option("display.max_rows", 10)

In [3]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": -7.38,
	"longitude": 112.7851,
	"start_date": "2024-12-30",
	"end_date": "2025-09-01",
	"hourly": "weather_code",
	"timezone": "auto",
}
responses = openmeteo.weather_api(url, params=params)
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

hourly = response.Hourly()
hourly_weather_code = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["weather_code"] = hourly_weather_code
hourly_dataframe = pd.DataFrame(data = hourly_data)

Coordinates: -7.375°N 112.75°E
Elevation: 2.0 m asl
Timezone: b'Asia/Jakarta'b'GMT+7'
Timezone difference to GMT+0: 25200s


In [4]:
hourly_dataframe.isna().sum()

date            0
weather_code    0
dtype: int64

In [5]:
d = pd.read_csv("d1.csv")
t = pd.read_csv("d2.csv")

In [6]:
display(d.info())
display(t.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127 entries, 0 to 126
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   tanggal              127 non-null    object 
 1   jam                  117 non-null    object 
 2   waktu                0 non-null      float64
 3   fase                 125 non-null    object 
 4   lokasi_perimeter     125 non-null    object 
 5   titik                49 non-null     float64
 6   kategori_kejadian    127 non-null    object 
 7   airline              127 non-null    object 
 8   runway_use           100 non-null    float64
 9   komponen_pesawat     59 non-null     object 
 10  dampak_pada_pesawat  29 non-null     object 
 11  kondisi_kerusakan    68 non-null     object 
 12  tindakan_perbaikan   39 non-null     object 
 13  sumber_informasi     113 non-null    object 
 14  remark               127 non-null    object 
 15  deskripsi            112 non-null    obj

None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30181 entries, 0 to 30180
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   no                    30181 non-null  int64  
 1   act_type              30158 non-null  object 
 2   reg_no                30152 non-null  object 
 3   opr                   30181 non-null  object 
 4   flight_number_origin  30144 non-null  object 
 5   flight_number_dest    30175 non-null  object 
 6   ata                   30181 non-null  object 
 7   block_on              30181 non-null  object 
 8   block_off             30181 non-null  object 
 9   atd                   30181 non-null  object 
 10  ground_time           30138 non-null  object 
 11  org                   30144 non-null  object 
 12  des                   30175 non-null  object 
 13  ps                    30144 non-null  object 
 14  runway                30147 non-null  float64
 15  avio_a             

None

#### <b>preprocess data bird strike

> Filter

In [7]:
# Filter & Validation

cols = [
    "tanggal",
    "jam",
    "waktu",
    "cuaca",
    "jumlah burung pada titik x",
    "titik",
    "fase",
    "strike",
]

df=(d[(pd.to_datetime(d["tanggal"], errors="coerce")).dt.year == 2025])
df = df[df["remark"] == "Terkonfirmasi"]
df = df[df["fase"].isin(["Landing", "Take Off"])]
df = df[df["runway_use"].isin([10, 28])]

def waktu_from_hour(h): 
    if pd.isna(h): return np.nan 
    if 0 <= h <= 3: return "Dini Hari" 
    if 3 < h <= 8: return "Pagi" 
    if 8 < h <= 13: return "Siang" 
    if 13 < h <= 18: return "Sore" 
    return "Malam"

jam_hour = pd.to_datetime(df.get("jam"), errors="coerce").dt.hour 
waktu = jam_hour.apply(waktu_from_hour)

a = pd.DataFrame(
    {
        "tanggal": df["tanggal"],
        "jam": df["jam"],
        "waktu": waktu,
        "cuaca": np.nan,                     
        "jumlah burung pada titik x": np.nan,  
        "titik": df["titik"],               
        "fase": df["fase"],
        "strike": 1,
    },
    columns=cols,
).reset_index(drop=True)

print("A.shape:", a.shape)
a.head(5)

A.shape: (42, 8)


,tanggal,jam,waktu,cuaca,jumlah burung pada titik x,titik,fase,strike
0,2025-01-06T00:00:00.000Z,1970-01-01T18:17:00.000Z,Sore,NaN,NaN,1.0,Landing,1
1,2025-02-09T00:00:00.000Z,1970-01-01T16:10:00.000Z,Sore,NaN,NaN,8.0,Landing,1
2,2025-02-21T00:00:00.000Z,1970-01-01T10:18:00.000Z,Siang,NaN,NaN,2.0,Take Off,1
3,2025-01-01T00:00:00.000Z,1970-01-01T06:33:00.000Z,Pagi,NaN,NaN,3.0,Landing,1
4,2025-02-25T00:00:00.000Z,1970-01-01T12:55:00.000Z,Siang,NaN,NaN,7.0,Landing,1


> Fill Cuaca

In [8]:
hourly_dataframe['date'] = pd.to_datetime(hourly_dataframe['date'], errors='coerce')
if hourly_dataframe['date'].dt.tz is not None:
    hourly_dataframe['date'] = hourly_dataframe['date'].dt.tz_convert(None)
hourly_dataframe['dt_hour'] = hourly_dataframe['date'].dt.floor('H')
hourly_dataframe['key_hour'] = hourly_dataframe['dt_hour'].dt.strftime('%Y-%m-%d %H:%M')

k = a.copy()
k['jam_parsed'] = pd.to_datetime(k['jam'], format='%H:%M', errors='coerce')
if k['jam_parsed'].isna().any():
    k['jam_parsed'] = pd.to_datetime(k['jam'], errors='coerce').dt.time

def jam_to_hm(x):
    if pd.isna(x):
        return np.nan
    # if it's Timestamp -> extract time
    if isinstance(x, pd.Timestamp):
        return x.strftime('%H:%M')
    # if it's time object
    try:
        return x.strftime('%H:%M')
    except Exception:
        return np.nan

k['jam_hm'] = k['jam_parsed'].apply(jam_to_hm)
k['tanggal_parsed'] = pd.to_datetime(k['tanggal'], errors='coerce')
base_dt = hourly_dataframe['date'].min()
if pd.isna(base_dt):
    raise ValueError("hourly_dataframe['date'] looks empty or couldn't be parsed.")

base_year = base_dt.year
base_month = base_dt.month

def build_dt(row):
    jam = row['jam_hm']
    # if we have full date
    if pd.notna(row['tanggal_parsed']):
        try:
            date_only = row['tanggal_parsed'].date()
            if pd.notna(jam):
                return pd.to_datetime(f"{date_only} {jam}", errors='coerce').floor('H')
            else:
                return pd.to_datetime(f"{date_only} 00:00", errors='coerce').floor('H')
        except Exception:
            return pd.NaT
    try:
        day = int(row['tanggal'])
        # use base year-month (this assumes your dataset covers the same month)
        date_str = f"{base_year}-{base_month:02d}-{day:02d}"
        if pd.notna(jam):
            return pd.to_datetime(f"{date_str} {jam}", errors='coerce').floor('H')
        else:
            return pd.to_datetime(f"{date_str} 00:00", errors='coerce').floor('H')
    except Exception:
        return pd.NaT

k['dt_hour'] = k.apply(build_dt, axis=1)

# optional key string
k['key_hour'] = k['dt_hour'].dt.strftime('%Y-%m-%d %H:%M')

# --- Merge on full datetime key ---
merged = k.merge(
    hourly_dataframe[['dt_hour', 'weather_code', 'date']],
    on='dt_hour',
    how='left',
    validate='m:1'  # many rows in k may map to same hourly row
)

# --- Map weather_code to description (same as before) ---
def map_to_4_cuaca(code):
    mapping = {
        0:"Cerah", 1:"Cerah Berawan", 2:"Berawan Sebagian", 3:"Berawan",
        45:"Berkabut", 48:"Rime Kabut",
        51:"Gerimis Ringan", 53:"Gerimis Sedang", 55:"Gerimis Lebat",
        61:"Hujan Ringan", 63:"Hujan Sedang", 65:"Hujan Lebat",
        80:"Hujan Gerimis", 81:"Hujan Lebat Sesaat", 82:"Hujan Sangat Lebat Sesaat",
        95:"Badai Petir", 96:"Badai Petir", 99:"Badai Petir"
    }
    if pd.isna(code):
        return np.nan
    try:
        return mapping.get(int(code), f"Kode {int(code)}")
    except Exception:
        return np.nan

merged['cuaca'] = merged['weather_code'].apply(map_to_4_cuaca).fillna('Tidak tersedia')

# --- Debug: show rows that didn't match (dt_hour is NaT or weather_code is NaN) ---
no_dt = merged[merged['dt_hour'].isna()].head(10)
no_weather = merged[merged['weather_code'].isna() & merged['dt_hour'].notna()].head(10)

# final df
k_filled = merged.drop(columns=['weather_code','date'])
k_filled = k_filled.rename(columns={'jam': 'jam_asli'})  # optional


In [9]:
a = k_filled.drop(columns=["jam_parsed", "jam_hm", "tanggal_parsed", "dt_hour", "key_hour"])
a = a.rename(columns={"jam_asli": "jam"})
a.head()

,tanggal,jam,waktu,cuaca,jumlah burung pada titik x,titik,fase,strike
0,2025-01-06T00:00:00.000Z,1970-01-01T18:17:00.000Z,Sore,Berawan,NaN,1.0,Landing,1
1,2025-02-09T00:00:00.000Z,1970-01-01T16:10:00.000Z,Sore,Berawan,NaN,8.0,Landing,1
2,2025-02-21T00:00:00.000Z,1970-01-01T10:18:00.000Z,Siang,Hujan Gerimis,NaN,2.0,Take Off,1
3,2025-01-01T00:00:00.000Z,1970-01-01T06:33:00.000Z,Pagi,Badai Petir,NaN,3.0,Landing,1
4,2025-02-25T00:00:00.000Z,1970-01-01T12:55:00.000Z,Siang,Hujan Gerimis,NaN,7.0,Landing,1


#### <b>preprocess data flight

In [10]:
folder_path = "modeling"
all_files = sorted(glob.glob(os.path.join(folder_path, "*.csv")))

def normalize_text_file(path, encoding="utf-8"):
    out_lines = []
    with open(path, "r", encoding=encoding, errors="replace") as f:
        for raw in f:
            line = raw.rstrip("\n\r")
            if len(line) >= 2 and line.startswith('"') and line.endswith('"'):
                # hilangkan pembungkus luar dan ganti `""` -> `"`
                line = line[1:-1].replace('""', '"')
            out_lines.append(line)
    return "\n".join(out_lines)

# header tujuan (sesuai yang kamu minta)
desired_columns = [
    "no", "act_type", "reg_no", "opr", "flight_number_origin", "flight_number_dest",
    "ata", "block_on", "block_off", "atd", "ground_time", "org", "des", "ps",
    "runway", "avio_a", "avio_d", "f_stat", "bulan", "tahun"
]

def csv_splitter(line):
    # gunakan pandas untuk split aman satu baris CSV
    return pd.read_csv(io.StringIO(line), header=None, quotechar='"', engine="python").astype(str).iloc[0].tolist()

def looks_like_header_line_first(line, desired_columns):
    # deteksi header pada baris pertama file (lebih konservatif)
    try:
        cells = [c.strip() for c in csv_splitter(line)]
    except Exception:
        return False
    lower_cells = [c.lower() for c in cells]
    desired_lower = [d.lower() for d in desired_columns]
    # jika salah satu cell cocok dengan salah satu desired_columns -> header
    if any(c in desired_lower for c in lower_cells):
        return True
    # jika mayoritas cell mengandung huruf (bukan angka) -> kemungkinan header
    alpha_count = sum(1 for c in cells if re.search(r'[A-Za-z]', c))
    if len(cells) > 0 and (alpha_count / len(cells)) >= 0.6:
        return True
    return False

df_list = []
for file in all_files:
    text = normalize_text_file(file)
    if not text.strip():
        # file kosong -> lewati
        continue
    lines = text.splitlines()

    # tentukan apakah baris pertama adalah header -> hapus jika ya
    if lines:
        first_line = lines[0]
        if looks_like_header_line_first(first_line, desired_columns):
            lines = lines[1:]

    # baca sebagai data tanpa header
    text_no_header = "\n".join(lines)
    if text_no_header.strip() == "":
        df = pd.DataFrame()
    else:
        df = pd.read_csv(io.StringIO(text_no_header), header=None, quotechar='"', engine="python", dtype=str)
    df_list.append(df)

# gabungkan semua
if df_list:
    merged = pd.concat(df_list, ignore_index=True, sort=False)
else:
    merged = pd.DataFrame()

# --- Deteksi & hapus header-like rows di SELURUH merged (bukan cuma baris pertama) ---
if not merged.empty:
    n_desired = len(desired_columns)
    n_actual = merged.shape[1]
    # prepare set of desired lower names for membership test
    desired_set = set([c.lower() for c in desired_columns])

    # untuk tiap baris, hitung berapa sel yang "match" (cell lower == some desired column)
    # serta hitung proporsi match terhadap jumlah kolom yang ada pada baris tersebut (n_actual)
    def row_header_score(row):
        count = 0
        for v in row:
            try:
                s = ("" if pd.isna(v) else str(v)).strip().lower()
            except Exception:
                s = ""
            if s in desired_set:
                count += 1
        return count

    match_counts = merged.apply(row_header_score, axis=1)
    # threshold gunakan 60% dari min(n_actual, n_desired) agar adil saat jumlah kolom berbeda
    k = min(n_actual if n_actual > 0 else 1, n_desired)
    threshold = max(1, math.ceil(0.6 * k))

    header_mask = match_counts >= threshold

    # debug: berapa baris yang terdeteksi header
    n_header_rows = int(header_mask.sum())
    if n_header_rows:
        print(f"Detected and removing {n_header_rows} header-like row(s) inside merged data (threshold={threshold}).")
    # hapus baris header-like
    merged = merged.loc[~header_mask].reset_index(drop=True)

# sesuaikan jumlah kolom: tambah kolom kosong atau potong sesuai panjang desired_columns
n_desired = len(desired_columns)
n_actual = merged.shape[1]

if n_actual < n_desired:
    for i in range(n_actual, n_desired):
        merged[i] = ""
    print(f"Warning: files punya {n_actual} kolom, ditambah menjadi {n_desired} kolom sesuai header tujuan.")
elif n_actual > n_desired:
    merged = merged.iloc[:, :n_desired]
    print(f"Warning: files punya {n_actual} kolom, dipotong menjadi {n_desired} kolom sesuai header tujuan.")

# beri nama kolom sesuai desired_columns
merged.columns = desired_columns

# bersihkan whitespace di string kolom
merged = merged.applymap(lambda x: x.strip() if isinstance(x, str) else x)

# perbaiki/atur kolom 'no' jadi 1..N (overwrite kolom no)
merged["no"] = range(1, len(merged) + 1)

# hasil akhir
m = merged
m

C:\Users\nalin\AppData\Local\Temp\ipykernel_28844\4276890416.py:121: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  merged = merged.applymap(lambda x: x.strip() if isinstance(x, str) else x)


,no,act_type,reg_no,opr,flight_number_origin,flight_number_dest,ata,block_on,block_off,atd,ground_time,org,des,ps,runway,avio_a,avio_d,f_stat,bulan,tahun
0,1,B738,PKLKP,LNI,LNI681,LNI878,31/15:51,31/15:56,01/04:59,01/05:07,13:02:08,PKY,AMQ,18,28,0,0,NML,1,2025
1,2,A320,PKAZQ,AWQ,AWQ327,AWQ320,01/00:03,01/00:07,01/05:03,01/05:13,4:56:00,KUL,KUL,A03,28,1,1,NML,1,2025
2,3,A320,PKPWD,PAS,PAS212,PAS213,31/20:31,31/20:37,01/05:40,01/05:48,9:03:00,CGK,CGK,5,28,1,1,NML,1,2025
3,4,A320,PKGLL,CTV,CTV695,CTV725,31/22:47,31/22:56,01/05:38,01/05:46,6:42:00,DPS,CGK,7,28,1,1,NML,1,2025
4,5,B739,PKLHY,LNI,LNI983,LNI882,31/13:44,31/13:48,01/06:03,01/06:09,16:14:43,PKU,UPG,23,28,0,0,NML,1,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30176,30177,A320,PKGQL,CTV,CTV736,CTV352,31/19:16,31/19:20,31/20:12,31/20:23,0:52:00,CGK,UPG,008,10,1,1,NML,8,2025
30177,30178,A320,PKGQJ,CTV,CTV949,CTV737,31/18:58,31/19:03,31/20:36,31/20:48,1:33:00,BTH,CGK,007,10,1,1,NML,8,2025
30178,30179,B739,PKLHI,LNI,LNI923,LNI880,31/23:28,31/23:31,01/00:13,01/00:25,0:42:40,DPS,UPG,012,10,1,1,NML,8,2025
30179,30180,B739,PKLHY,LNI,LNI709,LNI581,31/21:51,31/21:56,31/22:48,31/23:00,0:52:00,UPG,CGK,014,10,1,1,NML,8,2025


In [11]:
# NaN

pd.set_option("display.max_rows",None)
print("Data NaN:",m.isna().sum().sum())
print(m.isna().sum())
m[m["ground_time"].isna()].head(5)

Data NaN: 147
no                       0
act_type                23
reg_no                  29
opr                      0
flight_number_origin     0
flight_number_dest       6
ata                      0
block_on                 0
block_off                0
atd                      0
ground_time             43
org                      0
des                      0
ps                      37
runway                   0
avio_a                   0
avio_d                   0
f_stat                   9
bulan                    0
tahun                    0
dtype: int64


,no,act_type,reg_no,opr,flight_number_origin,flight_number_dest,ata,block_on,block_off,atd,ground_time,org,des,ps,runway,avio_a,avio_d,f_stat,bulan,tahun
1291,1292,B739,PKLSH,LNI,-,LNI266,--:--,--:--,12/05:59,--:--,NaN,-,BPN,22,28,0,-,NML,1,2025
1900,1901,NaN,NaN,SUS,-,SUS6197,--:--,--:--,--:--,--:--,NaN,-,BXW,NaN,-,-1,-,NML,1,2025
3919,3920,B733F,NaN,MYU,-,MYU815,--:--,--:--,03/05:01,03/05:15,NaN,-,CGK,C02,28,0,-,NML,2,2025
4204,4205,NaN,NaN,TGW,-,TGW265,--:--,--:--,--:--,--:--,NaN,-,SIN,NaN,28,-1,-,NML,2,2025
5651,5652,NaN,NaN,MYU,-,MYU815,--:--,--:--,--:--,--:--,NaN,-,CGK,NaN,-,-1,-,NML,2,2025


In [12]:
pd.set_option("display.max_rows", 10)

m = m.dropna()
m = m[~m.apply(lambda row: row.astype(str).str.contains("--:--").any(), axis=1)]
m = m.reset_index(drop=True)
m["no"] = m.index + 1

m

,no,act_type,reg_no,opr,flight_number_origin,flight_number_dest,ata,block_on,block_off,atd,ground_time,org,des,ps,runway,avio_a,avio_d,f_stat,bulan,tahun
0,1,B738,PKLKP,LNI,LNI681,LNI878,31/15:51,31/15:56,01/04:59,01/05:07,13:02:08,PKY,AMQ,18,28,0,0,NML,1,2025
1,2,A320,PKAZQ,AWQ,AWQ327,AWQ320,01/00:03,01/00:07,01/05:03,01/05:13,4:56:00,KUL,KUL,A03,28,1,1,NML,1,2025
2,3,A320,PKPWD,PAS,PAS212,PAS213,31/20:31,31/20:37,01/05:40,01/05:48,9:03:00,CGK,CGK,5,28,1,1,NML,1,2025
3,4,A320,PKGLL,CTV,CTV695,CTV725,31/22:47,31/22:56,01/05:38,01/05:46,6:42:00,DPS,CGK,7,28,1,1,NML,1,2025
4,5,B739,PKLHY,LNI,LNI983,LNI882,31/13:44,31/13:48,01/06:03,01/06:09,16:14:43,PKU,UPG,23,28,0,0,NML,1,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30129,30130,A320,PKGQL,CTV,CTV736,CTV352,31/19:16,31/19:20,31/20:12,31/20:23,0:52:00,CGK,UPG,008,10,1,1,NML,8,2025
30130,30131,A320,PKGQJ,CTV,CTV949,CTV737,31/18:58,31/19:03,31/20:36,31/20:48,1:33:00,BTH,CGK,007,10,1,1,NML,8,2025
30131,30132,B739,PKLHI,LNI,LNI923,LNI880,31/23:28,31/23:31,01/00:13,01/00:25,0:42:40,DPS,UPG,012,10,1,1,NML,8,2025
30132,30133,B739,PKLHY,LNI,LNI709,LNI581,31/21:51,31/21:56,31/22:48,31/23:00,0:52:00,UPG,CGK,014,10,1,1,NML,8,2025


In [13]:
import re
import pandas as pd
import numpy as np

# --------------------------------------------------
# Fungsi utilitas
# --------------------------------------------------

def extract_day_and_time(s):
    """
    Input: string yang diharapkan mengandung pattern "day / time" (contoh: "12 / 08:30" atau "5/8:30:00")
    Output: (day_str_or_None, jam_str_or_None)
    """
    if not isinstance(s, str) or "/" not in s:
        return (None, None)
    left, right = s.split("/", 1)
    left = left.strip()
    right = right.strip()

    # cari pola jam (ambil jam terakhir jika ada banyak teks)
    m = re.search(r'(\d{1,2}:\d{2}(?::\d{2})?)', right)
    if m:
        jam = m.group(1)
    else:
        jam = right if re.match(r'^\d{1,2}:\d{2}(:\d{2})?$', right) else None

    day_out = left if left != "" else None
    return (day_out, jam)

def waktu_from_hour(h):
    """
    h: integer hour (0-23) atau NaN
    return: kategori waktu (string) atau np.nan
    """
    if pd.isna(h):
        return np.nan
    try:
        h = int(h)
    except Exception:
        return np.nan
    if 0 <= h <= 3:
        return "Dini Hari"
    if 3 < h <= 8:
        return "Pagi"
    if 8 < h <= 13:
        return "Siang"
    if 13 < h <= 18:
        return "Sore"
    return "Malam"

# --------------------------------------------------
# Input: dataframe sumber
# Asumsi: dataframe asal bernama `m` sudah ada di environment,
#        dan memiliki kolom minimal: 'bulan', 'tahun', 'no', 'ata', 'atd'
# --------------------------------------------------

df_input = m  # ganti jika nama berbeda

rows = []
for idx, r in df_input.iterrows():
    bulan = r.get("bulan")
    tahun = r.get("tahun")
    source_no = r.get("no")  # nomor baris asal

    # ---------- Landing (ATA) ----------
    ata_val = r.get("ata")
    day_str, jam_str = extract_day_and_time(ata_val)
    if day_str is not None and jam_str is not None:
        # bersihkan day dan konversi ke integer bila memungkinkan
        try:
            day_int = int(re.sub(r'\D', '', day_str))
        except Exception:
            day_int = None

        # catat row tapi tanpa kolom 'tanggal' lengkap
        rows.append({
            "day": day_int,
            "month": int(bulan) if pd.notna(bulan) else None,
            "year": int(tahun) if pd.notna(tahun) else None,
            "jam": jam_str,
            "waktu": np.nan,
            "cuaca": np.nan,
            "jumlah burung pada titik x": np.nan,
            "titik": np.nan,
            "fase": "Landing",
            "strike": 0,
            "source_no": source_no
        })

    # ---------- Take Off (ATD) ----------
    atd_val = r.get("atd")
    day2_str, jam2_str = extract_day_and_time(atd_val)
    if day2_str is not None and jam2_str is not None:
        try:
            day2_int = int(re.sub(r'\D', '', day2_str))
        except Exception:
            day2_int = None

        rows.append({
            "day": day2_int,
            "month": int(bulan) if pd.notna(bulan) else None,
            "year": int(tahun) if pd.notna(tahun) else None,
            "jam": jam2_str,
            "waktu": np.nan,
            "cuaca": np.nan,
            "jumlah burung pada titik x": np.nan,
            "titik": np.nan,
            "fase": "Take Off",
            "strike": 0,
            "source_no": source_no
        })

# --------------------------------------------------
# Buat dataframe final (tidak ada kolom 'tanggal')
# --------------------------------------------------

cols = [
    "day", "month", "year", "jam", "waktu", "cuaca",
    "jumlah burung pada titik x", "titik", "fase", "strike", "source_no"
]
df_final = pd.DataFrame(rows, columns=cols)

# jika ada baris, isi kolom waktu berdasarkan jam dan set tipe day/month/year ke Int64 nullable
if not df_final.empty:
    # hitung jam dari kolom jam (jika bisa)
    jam_hour = pd.to_datetime(df_final["jam"], errors="coerce").dt.hour
    df_final["waktu"] = jam_hour.apply(waktu_from_hour)

    # ubah day/month/year ke tipe integer nullable agar menampung NA
    for c in ["day", "month", "year"]:
        # jika kolom ada, ubah ke numeric lalu ke Int64 (nullable)
        if c in df_final.columns:
            df_final[c] = pd.to_numeric(df_final[c], errors="coerce").astype("Int64")

# reset index
df_final = df_final.reset_index(drop=True)

# hasil akhir
j = df_final
j

C:\Users\nalin\AppData\Local\Temp\ipykernel_28844\2683872706.py:126: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  jam_hour = pd.to_datetime(df_final["jam"], errors="coerce").dt.hour


,day,month,year,jam,waktu,cuaca,jumlah burung pada titik x,titik,fase,strike,source_no
0,31,1,2025,15:51,Sore,NaN,NaN,NaN,Landing,0,1
1,1,1,2025,05:07,Pagi,NaN,NaN,NaN,Take Off,0,1
2,1,1,2025,00:03,Dini Hari,NaN,NaN,NaN,Landing,0,2
3,1,1,2025,05:13,Pagi,NaN,NaN,NaN,Take Off,0,2
4,31,1,2025,20:31,Malam,NaN,NaN,NaN,Landing,0,3
...,...,...,...,...,...,...,...,...,...,...,...
60263,1,8,2025,00:25,Dini Hari,NaN,NaN,NaN,Take Off,0,30132
60264,31,8,2025,21:51,Malam,NaN,NaN,NaN,Landing,0,30133
60265,31,8,2025,23:00,Malam,NaN,NaN,NaN,Take Off,0,30133
60266,31,8,2025,20:54,Malam,NaN,NaN,NaN,Landing,0,30134


In [14]:
import calendar
import pandas as pd
import numpy as np

def safe_int(x):
    """Convert pandas-y values (Int64/float/NA/str) to int or None."""
    try:
        if pd.isna(x):
            return None
        return int(x)
    except Exception:
        return None

def max_day(year, month):
    """Return maximum day for (year, month) or np.nan on error."""
    try:
        return calendar.monthrange(int(year), int(month))[1]
    except Exception:
        return np.nan

def build_tanggal_cap_from_row(r):
    """
    Build a pd.Timestamp from row with keys 'year','month','day'.
    If day > max_day(year,month) we cap to that max_day.
    Returns pd.NaT on failure.
    """
    y = safe_int(r.get("year"))
    m = safe_int(r.get("month"))
    d = safe_int(r.get("day"))
    if any(v is None for v in (y, m, d)):
        return pd.NaT
    try:
        md = max_day(y, m)
        if pd.isna(md):
            return pd.NaT
        d_used = min(d, int(md))
        return pd.Timestamp(year=y, month=m, day=d_used)
    except Exception:
        return pd.NaT

def apply_validasi_manual_v2(df):
    """
    Refactored validation function.
    Input: DataFrame with columns at least ['source_no','year','month','day','fase'].
    Output: (df_out, dropped_df, fixed_df)
      - df_out: dataframe after applying month fixes and dropping marked rows; index reset
      - dropped_df: copy of rows that were dropped (index preserved from original)
      - fixed_df: copy of rows that were candidate-fixed (before applying changes)
    """
    df_out = df.copy()
    rows_to_drop = []      # store original index values to drop
    rows_to_fix = []       # store tuples (orig_index, new_month)
    fixed_candidates_idx = []  # indices of rows that will be in fixed_df (for reporting/debug)

    # iterate per source_no preserving appearance order
    for source, group in df_out.groupby("source_no", sort=False):
        # reset_index to keep track of original indices via 'index' column
        grp = group.reset_index(drop=False)  # now grp['index'] is original df index

        for i in range(len(grp) - 1):
            cur = grp.loc[i]
            nxt = grp.loc[i + 1]

            # normalize fase text
            cur_fase = str(cur.get("fase")).strip().lower() if pd.notna(cur.get("fase")) else ""
            nxt_fase = str(nxt.get("fase")).strip().lower() if pd.notna(nxt.get("fase")) else ""
            if cur_fase != "landing" or nxt_fase != "take off":
                continue

            # safe ints for date components
            y = safe_int(cur.get("year")); m = safe_int(cur.get("month")); d = safe_int(cur.get("day"))
            ny = safe_int(nxt.get("year")); nm = safe_int(nxt.get("month")); nd = safe_int(nxt.get("day"))
            if any(v is None for v in [y, m, d, ny, nm, nd]):
                continue

            # require next (take off) day == 1
            if nd != 1:
                continue

            # next month can be same month or the following month with year rollover
            next_month = m + 1 if m < 12 else 1
            next_year = y if m < 12 else y + 1
            if not ((nm == m and ny == y) or (nm == next_month and ny == next_year)):
                continue

            orig_idx = cur["index"]  # original index in df

            # Apply rules (refined & consolidated)
            # bulan 01: day in [29,30,31] -> drop
            if m == 1 and d in (29, 30, 31):
                rows_to_drop.append(orig_idx)
                continue

            # bulan 02: day in [29,30,31] -> change month -> 1
            if m == 2 and d in (29, 30, 31):
                rows_to_fix.append((orig_idx, 1))
                fixed_candidates_idx.append(orig_idx)
                continue

            # bulan 03: odd/even year logic
            if m == 3:
                if (y % 2 == 1 and d == 28) or (y % 2 == 0 and d in (28, 29)):
                    rows_to_fix.append((orig_idx, 2))
                    fixed_candidates_idx.append(orig_idx)
                continue

            # bulan 04: if d >= max_day(bulan 3) -> change to bulan 3
            if m == 4:
                md_prev = max_day(y, 3)
                if not pd.isna(md_prev) and d >= md_prev:
                    rows_to_fix.append((orig_idx, 3))
                    fixed_candidates_idx.append(orig_idx)
                continue

            # bulan 05: if d >= max_day(bulan 4) -> change to bulan 4
            if m == 5:
                md_prev = max_day(y, 4)
                if not pd.isna(md_prev) and d >= md_prev:
                    rows_to_fix.append((orig_idx, 4))
                    fixed_candidates_idx.append(orig_idx)
                continue

            # bulan 06..12: if d >= max_day(bulan m-1) -> change to bulan m-1
            if 6 <= m <= 12:
                md_prev = max_day(y, m-1)
                if not pd.isna(md_prev) and d >= md_prev:
                    rows_to_fix.append((orig_idx, m-1))
                    fixed_candidates_idx.append(orig_idx)
                continue

    # build dropped_df and fixed_df (before applying changes)
    dropped_df = df_out.loc[rows_to_drop].copy() if rows_to_drop else pd.DataFrame(columns=df_out.columns)
    fixed_df = df_out.loc[fixed_candidates_idx].copy() if fixed_candidates_idx else pd.DataFrame(columns=df_out.columns)

    # apply month fixes (mutate month) - only if index present
    for orig_idx, new_m in rows_to_fix:
        if orig_idx in df_out.index:
            df_out.loc[orig_idx, "month"] = new_m

    # drop rows and reset index
    df_out = df_out.drop(rows_to_drop, errors="ignore").reset_index(drop=True)

    # rebuild 'tanggal' column by capping invalid days
    # if original had 'day','month','year' columns, we create/overwrite 'tanggal'
    if set(("year","month","day")).issubset(df_out.columns):
        df_out["tanggal"] = df_out.apply(build_tanggal_cap_from_row, axis=1)
    else:
        # else keep existing 'tanggal' if any, or create NaT
        df_out["tanggal"] = pd.NaT

    return df_out, dropped_df.reset_index(drop=True), fixed_df.reset_index(drop=True)

In [15]:
df_out, dropped_df, fixed_df = apply_validasi_manual_v2(j)

print("Dropped rows (sample):")
print(dropped_df.head())

print("\nFixed-candidates (before change):")
print(fixed_df.head())

print("\nResult sample:")
print(df_out.loc[:10])


Dropped rows (sample):
   day  month  year    jam  waktu  cuaca  jumlah burung pada titik x  titik  \
0   31      1  2025  15:51   Sore    NaN                         NaN    NaN   
1   31      1  2025  20:31  Malam    NaN                         NaN    NaN   
2   31      1  2025  22:47  Malam    NaN                         NaN    NaN   
3   31      1  2025  13:44  Siang    NaN                         NaN    NaN   
4   31      1  2025  15:18   Sore    NaN                         NaN    NaN   

      fase  strike  source_no  
0  Landing       0          1  
1  Landing       0          3  
2  Landing       0          4  
3  Landing       0          5  
4  Landing       0          6  

Fixed-candidates (before change):
   day  month  year    jam  waktu  cuaca  jumlah burung pada titik x  titik  \
0   31      2  2025  22:29  Malam    NaN                         NaN    NaN   
1   31      2  2025  20:19  Malam    NaN                         NaN    NaN   
2   31      2  2025  12:19  Siang    N

In [16]:
j_valid = df_out

In [17]:
import calendar
import numpy as np
import pandas as pd

# pastikan kolom year/month/day dalam numeric (NaN kalau tidak bisa)
y = pd.to_numeric(j_valid['year'], errors='coerce').astype('Int64')
m = pd.to_numeric(j_valid['month'], errors='coerce').astype('Int64')
d = pd.to_numeric(j_valid['day'], errors='coerce').astype('Int64')

# fungsi untuk vectorized monthrange (list comprehension)
def max_days_list(years, months):
    md = []
    for yy, mm in zip(years, months):
        try:
            if pd.isna(yy) or pd.isna(mm):
                md.append(np.nan)
            else:
                md.append(calendar.monthrange(int(yy), int(mm))[1])
        except Exception:
            md.append(np.nan)
    return np.array(md, dtype='float64')

md_arr = max_days_list(y.values, m.values)

# cap day ke max_day (jika day atau max_day NaN → tetap NaN)
day_capped = np.where(np.isnan(md_arr) | pd.isna(d.values), np.nan, np.minimum(d.values, md_arr)).astype('float64')

# buat tanggal dari year/month/day_capped
j_valid['tanggal'] = pd.to_datetime(
    pd.DataFrame({
        'year': y.astype('float64'),    # pd.to_datetime menerima float-ish ints
        'month': m.astype('float64'),
        'day': day_capped
    }),
    errors='coerce'
)

# (opsional) cek baris yang sebelumnya non-null tapi sekarang NaT
problem_mask = j_valid['tanggal'].isna() & j_valid[['year','month','day']].notna().all(axis=1)
if problem_mask.any():
    print("Baris bermasalah (input valid tapi pd.to_datetime gagal even after capping):")
    print(j_valid.loc[problem_mask, ['year','month','day']].head())

# lalu drop kolom lama jika mau
j_valid = j_valid.drop(columns=['day','month','year','source_no'], errors='ignore')

# reorder
cols = ['tanggal'] + [c for c in j_valid.columns if c != 'tanggal']
j_valid = j_valid[cols]


In [18]:
j_valid.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60243 entries, 0 to 60242
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   tanggal                     60243 non-null  datetime64[ns]
 1   jam                         60243 non-null  object        
 2   waktu                       60243 non-null  object        
 3   cuaca                       0 non-null      float64       
 4   jumlah burung pada titik x  0 non-null      float64       
 5   titik                       0 non-null      float64       
 6   fase                        60243 non-null  object        
 7   strike                      60243 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(1), object(3)
memory usage: 3.7+ MB


In [19]:
hourly_dataframe['date'] = pd.to_datetime(hourly_dataframe['date'], errors='coerce')
if hourly_dataframe['date'].dt.tz is not None:
    hourly_dataframe['date'] = hourly_dataframe['date'].dt.tz_convert(None)
hourly_dataframe['dt_hour'] = hourly_dataframe['date'].dt.floor('H')
hourly_dataframe['key_hour'] = hourly_dataframe['dt_hour'].dt.strftime('%Y-%m-%d %H:%M')

j_valid = j_valid.copy()
j_valid['jam_parsed'] = pd.to_datetime(j_valid['jam'], format='%H:%M', errors='coerce')
if j_valid['jam_parsed'].isna().any():
    j_valid['jam_parsed'] = pd.to_datetime(j_valid['jam'], errors='coerce').dt.time

def jam_to_hm(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, pd.Timestamp):  # kalau Timestamp -> ambil time
        return x.strftime('%H:%M')
    try:  # kalau time object
        return x.strftime('%H:%M')
    except Exception:
        return np.nan

j_valid['jam_hm'] = j_valid['jam_parsed'].apply(jam_to_hm)
j_valid['tanggal_parsed'] = pd.to_datetime(j_valid['tanggal'], errors='coerce')

base_dt = hourly_dataframe['date'].min()
if pd.isna(base_dt):
    raise ValueError("hourly_dataframe['date'] looks empty or couldn't be parsed.")

base_year = base_dt.year
base_month = base_dt.month

def build_dt(row):
    jam = row['jam_hm']
    if pd.notna(row['tanggal_parsed']):  # kalau ada full date
        try:
            date_only = row['tanggal_parsed'].date()
            if pd.notna(jam):
                return pd.to_datetime(f"{date_only} {jam}", errors='coerce').floor('H')
            else:
                return pd.to_datetime(f"{date_only} 00:00", errors='coerce').floor('H')
        except Exception:
            return pd.NaT
    try:
        day = int(row['tanggal'])
        date_str = f"{base_year}-{base_month:02d}-{day:02d}"  # fallback
        if pd.notna(jam):
            return pd.to_datetime(f"{date_str} {jam}", errors='coerce').floor('H')
        else:
            return pd.to_datetime(f"{date_str} 00:00", errors='coerce').floor('H')
    except Exception:
        return pd.NaT

j_valid['dt_hour'] = j_valid.apply(build_dt, axis=1)

# buat key string
j_valid['key_hour'] = j_valid['dt_hour'].dt.strftime('%Y-%m-%d %H:%M')

# --- Merge ke hourly data ---
merged = j_valid.merge(
    hourly_dataframe[['dt_hour', 'weather_code', 'date']],
    on='dt_hour',
    how='left',
    validate='m:1'
)

# --- Mapping cuaca ---
def map_to_4_cuaca(code):
    mapping = {
        0:"Cerah", 1:"Cerah Berawan", 2:"Berawan Sebagian", 3:"Berawan",
        45:"Berkabut", 48:"Rime Kabut",
        51:"Gerimis Ringan", 53:"Gerimis Sedang", 55:"Gerimis Lebat",
        61:"Hujan Ringan", 63:"Hujan Sedang", 65:"Hujan Lebat",
        80:"Hujan Gerimis", 81:"Hujan Lebat Sesaat", 82:"Hujan Sangat Lebat Sesaat",
        95:"Badai Petir", 96:"Badai Petir", 99:"Badai Petir"
    }
    if pd.isna(code):
        return np.nan
    try:
        return mapping.get(int(code), f"Kode {int(code)}")
    except Exception:
        return np.nan

merged['cuaca'] = merged['weather_code'].apply(map_to_4_cuaca).fillna('Tidak tersedia')

# --- Debugging ---
no_dt = merged[merged['dt_hour'].isna()].head(10)
no_weather = merged[merged['weather_code'].isna() & merged['dt_hour'].notna()].head(10)

# --- Hasil akhir ---
j_valid = merged.drop(columns=['weather_code','date'])
j_valid = j_valid.rename(columns={'jam': 'jam_asli'}) 

In [20]:
n = j_valid.drop(columns=["jam_parsed", "jam_hm", "tanggal_parsed", "dt_hour", "key_hour"])
n = j_valid.rename(columns={"jam_asli": "jam"})
n.head()

,tanggal,jam,waktu,cuaca,jumlah burung pada titik x,titik,fase,strike,jam_parsed,jam_hm,tanggal_parsed,dt_hour,key_hour
0,2025-01-01,05:07,Pagi,Badai Petir,NaN,NaN,Take Off,0,1900-01-01 05:07:00,05:07,2025-01-01,2025-01-01 05:00:00,2025-01-01 05:00
1,2025-01-01,00:03,Dini Hari,Berawan,NaN,NaN,Landing,0,1900-01-01 00:03:00,00:03,2025-01-01,2025-01-01 00:00:00,2025-01-01 00:00
2,2025-01-01,05:13,Pagi,Badai Petir,NaN,NaN,Take Off,0,1900-01-01 05:13:00,05:13,2025-01-01,2025-01-01 05:00:00,2025-01-01 05:00
3,2025-01-01,05:48,Pagi,Badai Petir,NaN,NaN,Take Off,0,1900-01-01 05:48:00,05:48,2025-01-01,2025-01-01 05:00:00,2025-01-01 05:00
4,2025-01-01,05:46,Pagi,Badai Petir,NaN,NaN,Take Off,0,1900-01-01 05:46:00,05:46,2025-01-01,2025-01-01 05:00:00,2025-01-01 05:00


In [21]:
n = n.drop(columns=['jam_parsed', 'jam_hm', 'tanggal_parsed', 'dt_hour','key_hour'])

In [22]:
n

,tanggal,jam,waktu,cuaca,jumlah burung pada titik x,titik,fase,strike
0,2025-01-01,05:07,Pagi,Badai Petir,NaN,NaN,Take Off,0
1,2025-01-01,00:03,Dini Hari,Berawan,NaN,NaN,Landing,0
2,2025-01-01,05:13,Pagi,Badai Petir,NaN,NaN,Take Off,0
3,2025-01-01,05:48,Pagi,Badai Petir,NaN,NaN,Take Off,0
4,2025-01-01,05:46,Pagi,Badai Petir,NaN,NaN,Take Off,0
...,...,...,...,...,...,...,...,...
60238,2025-08-01,00:25,Dini Hari,Cerah Berawan,NaN,NaN,Take Off,0
60239,2025-08-31,21:51,Malam,Hujan Gerimis,NaN,NaN,Landing,0
60240,2025-08-31,23:00,Malam,Berawan Sebagian,NaN,NaN,Take Off,0
60241,2025-08-31,20:54,Malam,Berawan Sebagian,NaN,NaN,Landing,0


In [23]:
if 'titik' not in j_valid.columns:
    j_valid['titik'] = np.nan

# ulangi setiap baris 8 kali
j_expanded = j_valid.loc[j_valid.index.repeat(8)].copy().reset_index(drop=True)

# isi titik 1-8 berulang
j_expanded['titik'] = np.tile(np.arange(1, 9), len(j_valid))

In [24]:
j_expanded = j_expanded.rename(columns={'jam_asli': 'jam'})

j_expanded = j_expanded.drop(
    columns=['jam_parsed', 'jam_hm', 'tanggal_parsed', 'dt_hour', 'key_hour'],
    errors='ignore'  # biar aman kalau ada yg ga ada
)

In [25]:
j_expanded

,tanggal,jam,waktu,cuaca,jumlah burung pada titik x,titik,fase,strike
0,2025-01-01,05:07,Pagi,Badai Petir,NaN,1,Take Off,0
1,2025-01-01,05:07,Pagi,Badai Petir,NaN,2,Take Off,0
2,2025-01-01,05:07,Pagi,Badai Petir,NaN,3,Take Off,0
3,2025-01-01,05:07,Pagi,Badai Petir,NaN,4,Take Off,0
4,2025-01-01,05:07,Pagi,Badai Petir,NaN,5,Take Off,0
...,...,...,...,...,...,...,...,...
481939,2025-08-31,22:17,Malam,Hujan Gerimis,NaN,4,Take Off,0
481940,2025-08-31,22:17,Malam,Hujan Gerimis,NaN,5,Take Off,0
481941,2025-08-31,22:17,Malam,Hujan Gerimis,NaN,6,Take Off,0
481942,2025-08-31,22:17,Malam,Hujan Gerimis,NaN,7,Take Off,0


In [26]:
j_expanded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 481944 entries, 0 to 481943
Data columns (total 8 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   tanggal                     481944 non-null  datetime64[ns]
 1   jam                         481944 non-null  object        
 2   waktu                       481944 non-null  object        
 3   cuaca                       481944 non-null  object        
 4   jumlah burung pada titik x  0 non-null       float64       
 5   titik                       481944 non-null  int32         
 6   fase                        481944 non-null  object        
 7   strike                      481944 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int32(1), int64(1), object(4)
memory usage: 27.6+ MB


In [30]:
import pandas as pd

# Misalnya j_expanded adalah DataFrame yang sudah ada
# Gabungkan kolom 'tanggal' dan 'jam' menjadi satu kolom datetime
j_expanded['datetime_combined'] = pd.to_datetime(j_expanded['tanggal'].astype(str) + ' ' + j_expanded['jam'], errors='coerce')

# Format datetime_combined ke format ISO 8601 (misalnya 2025-01-21T16:03:00.000Z)
j_expanded['datetime_iso'] = j_expanded['datetime_combined'].dt.strftime('%Y-%m-%dT%H:%M:%S.') + j_expanded['datetime_combined'].dt.microsecond.astype(str).str[:3] + 'Z'

# Pisahkan kembali datetime_iso ke kolom 'tanggal' dan 'jam'
j_expanded['tanggal'] = j_expanded['datetime_iso'].str.split('T').str[0]  # Ambil bagian tanggal (YYYY-MM-DD)
j_expanded['jam'] = j_expanded['datetime_iso'].str.split('T').str[1].str[:-1]  # Ambil bagian jam (HH:MM:SS.sss)

# Hapus kolom datetime_combined dan datetime_iso jika tidak diperlukan
j_expanded = j_expanded.drop(columns=['datetime_combined', 'datetime_iso'])

# Tampilkan beberapa baris untuk memastikan hasilnya
j_expanded[['tanggal', 'jam']].head()


,tanggal,jam
0,2025-01-01,05:07:00.0
1,2025-01-01,05:07:00.0
2,2025-01-01,05:07:00.0
3,2025-01-01,05:07:00.0
4,2025-01-01,05:07:00.0


In [32]:
fix = pd.concat([a, j_expanded], ignore_index=True)

#### <b>modeling

In [33]:
fix.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 481986 entries, 0 to 481985
Data columns (total 8 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   tanggal                     481986 non-null  object 
 1   jam                         481986 non-null  object 
 2   waktu                       481986 non-null  object 
 3   cuaca                       481986 non-null  object 
 4   jumlah burung pada titik x  0 non-null       object 
 5   titik                       481986 non-null  float64
 6   fase                        481986 non-null  object 
 7   strike                      481986 non-null  int64  
dtypes: float64(1), int64(1), object(6)
memory usage: 29.4+ MB


In [34]:
fix.head()

,tanggal,jam,waktu,cuaca,jumlah burung pada titik x,titik,fase,strike
0,2025-01-06T00:00:00.000Z,1970-01-01T18:17:00.000Z,Sore,Berawan,NaN,1.0,Landing,1
1,2025-02-09T00:00:00.000Z,1970-01-01T16:10:00.000Z,Sore,Berawan,NaN,8.0,Landing,1
2,2025-02-21T00:00:00.000Z,1970-01-01T10:18:00.000Z,Siang,Hujan Gerimis,NaN,2.0,Take Off,1
3,2025-01-01T00:00:00.000Z,1970-01-01T06:33:00.000Z,Pagi,Badai Petir,NaN,3.0,Landing,1
4,2025-02-25T00:00:00.000Z,1970-01-01T12:55:00.000Z,Siang,Hujan Gerimis,NaN,7.0,Landing,1


> Isi Data Burung

In [35]:
b = pd.read_csv("bird.csv")

In [39]:
b.head(5)

,latitude,longitude,lokasi,titik,tanggal,jam,waktu,cuaca,jenis_burung,nama_ilmiah,jumlah_burung,keterangan,dokumentasi,createdAt,updatedAt,deletedAt
0,-7.3753,112.7703,Dalam,1,2025-07-15,08:26:00.0,Pagi,Cerah,Blekok Sawah,Ardeola speciosa,10,NaN,NaN,2025-09-04 11:57:00,NaN,NaN
1,-7.3753,112.7703,Dalam,1,2025-07-15,08:26:00.0,Pagi,Cerah,Bondol Peking,Lonchura punctulata,4,NaN,NaN,2025-09-04 11:57:00,NaN,NaN
2,-7.3753,112.7703,Dalam,1,2025-07-15,08:26:00.0,Pagi,Cerah,Cangak Abu,Ardea cinerea,1,NaN,NaN,2025-09-04 11:57:00,NaN,NaN
3,-7.3753,112.7703,Dalam,1,2025-07-15,08:26:00.0,Pagi,Cerah,Cangak Merah,Ardea purpurea,4,NaN,NaN,2025-09-04 11:57:00,NaN,NaN
4,-7.3753,112.7703,Dalam,1,2025-07-15,08:26:00.0,Pagi,Cerah,Kokokan Laut,Butorides striatus,1,NaN,NaN,2025-09-04 11:57:00,NaN,NaN


In [37]:
import pandas as pd

# Misalnya j_expanded adalah DataFrame yang sudah ada
# Gabungkan kolom 'tanggal' dan 'jam' menjadi satu kolom datetime
b['datetime_combined'] = pd.to_datetime(b['tanggal'].astype(str) + ' ' + b['jam'], errors='coerce')

# Format datetime_combined ke format ISO 8601 (misalnya 2025-01-21T16:03:00.000Z)
b['datetime_iso'] = b['datetime_combined'].dt.strftime('%Y-%m-%dT%H:%M:%S.') + b['datetime_combined'].dt.microsecond.astype(str).str[:3] + 'Z'

# Pisahkan kembali datetime_iso ke kolom 'tanggal' dan 'jam'
b['tanggal'] = b['datetime_iso'].str.split('T').str[0]  # Ambil bagian tanggal (YYYY-MM-DD)
b['jam'] = b['datetime_iso'].str.split('T').str[1].str[:-1]  # Ambil bagian jam (HH:MM:SS.sss)

# Hapus kolom datetime_combined dan datetime_iso jika tidak diperlukan
b = b.drop(columns=['datetime_combined', 'datetime_iso'])

# Tampilkan beberapa baris untuk memastikan hasilnya
b[['tanggal', 'jam']].head()

,tanggal,jam
0,2025-07-15,08:26:00.0
1,2025-07-15,08:26:00.0
2,2025-07-15,08:26:00.0
3,2025-07-15,08:26:00.0
4,2025-07-15,08:26:00.0


In [38]:
b.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 643 entries, 0 to 642
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   latitude       643 non-null    float64
 1   longitude      643 non-null    object 
 2   lokasi         643 non-null    object 
 3   titik          643 non-null    int64  
 4   tanggal        643 non-null    object 
 5   jam            643 non-null    object 
 6   waktu          643 non-null    object 
 7   cuaca          643 non-null    object 
 8   jenis_burung   643 non-null    object 
 9   nama_ilmiah    643 non-null    object 
 10  jumlah_burung  643 non-null    int64  
 11  keterangan     20 non-null     object 
 12  dokumentasi    0 non-null      float64
 13  createdAt      643 non-null    object 
 14  updatedAt      0 non-null      float64
 15  deletedAt      0 non-null      float64
dtypes: float64(4), int64(2), object(10)
memory usage: 80.5+ KB


In [61]:
import numpy as np
import pandas as pd

def fill_fix_with_b(df_fix, df_b, seed=42):
    """
    Mengisi df_fix['jumlah burung pada titik x'] dengan:
      - rata-rata kumulatif (expanding mean) dari df_b['jumlah_burung']
        per (titik, waktu) hingga tanggal baris df_fix (<= target)
      - khusus baris df_fix dengan tanggal lebih awal daripada tanggal minimum df_b
        pada grup (titik, waktu), diisi dengan mean_harian dari SATU tanggal acak
        (per grup) di df_b (bukan kumulatif).
    """
    df_fix = df_fix.copy()
    df_b = df_b.copy()

    target_col = 'jumlah burung pada titik x'
    if target_col not in df_fix.columns:
        df_fix[target_col] = np.nan

    # ==============================
    # 1) Normalisasi kolom penting
    # ==============================
    # Pastikan kolom-kolom wajib ada
    required_fix = {'titik', 'waktu', 'tanggal'}
    required_b = {'titik', 'waktu', 'tanggal', 'jumlah_burung'}
    
    missing_fix = required_fix - set(df_fix.columns)
    missing_b = required_b - set(df_b.columns)

    if missing_fix:
        raise KeyError(f"df_fix missing columns: {missing_fix}")
    if missing_b:
        raise KeyError(f"df_b missing columns: {missing_b}")

    # Parse tanggal ke datetime64[ns], lalu dinormalisasi ke tengah malam (00:00)
    df_fix['tanggal'] = pd.to_datetime(df_fix['tanggal'], errors='coerce')
    df_b['tanggal'] = pd.to_datetime(df_b['tanggal'], errors='coerce')

    # Hilangkan baris dengan tanggal NaT (opsional: bisa juga dipertahankan)
    df_fix = df_fix[df_fix['tanggal'].notna()].copy()
    df_b = df_b[df_b['tanggal'].notna()].copy()

    # Normalisasi waktu (judul: Pagi/Siang/Sore/Malam, dsb)
    df_fix['waktu'] = df_fix['waktu'].astype(str).str.strip().str.title()
    df_b['waktu'] = df_b['waktu'].astype(str).str.strip().str.title()

    # Samakan tipe 'titik' (df_fix float64, df_b int64 di data kamu)
    def to_Int64(s):
        s_num = pd.to_numeric(s, errors='coerce')
        return s_num.astype('Int64')

    df_fix['titik'] = to_Int64(df_fix['titik'])
    df_b['titik'] = to_Int64(df_b['titik'])

    # Pastikan jumlah_burung numeric
    df_b['jumlah_burung'] = pd.to_numeric(df_b['jumlah_burung'], errors='coerce')

    # Normalisasi tanggal ke date boundary (optional; merge_asof cukup butuh sorted datetime)
    df_fix['_tanggal_dt'] = df_fix['tanggal'].dt.normalize()
    df_b['_tanggal_dt'] = df_b['tanggal'].dt.normalize()

    # ==============================
    # 2) Mean harian & cumavg
    # ==============================
    # Hitung rata-rata per (titik, waktu, tanggal)
    mean_harian = (
        df_b
        .groupby(['titik', 'waktu', '_tanggal_dt'], as_index=False)['jumlah_burung']
        .mean()
        .rename(columns={'jumlah_burung': 'mean_harian'})
    )

    # Urutkan dan hitung rata-rata kumulatif (expanding mean) per (titik, waktu)
    mean_harian = mean_harian.sort_values(['titik', 'waktu', '_tanggal_dt'])
    mean_harian['cumavg'] = (
        mean_harian
        .groupby(['titik', 'waktu'], group_keys=False)['mean_harian']
        .apply(lambda s: s.expanding().mean())
    )

    # ==============================
    # 3) Merge_asof (≤ tanggal)
    # ==============================
    # Urutkan df_fix dan mean_harian berdasarkan kolom _tanggal_dt
    df_fix = df_fix.sort_values(['titik', 'waktu', '_tanggal_dt'])  # Pastikan df_fix terurut
    mean_harian = mean_harian.sort_values(['titik', 'waktu', '_tanggal_dt'])  # Pastikan mean_harian terurut

    # Lakukan merge_asof setelah urutan benar
    merged = pd.merge_asof(
        left=df_fix,
        right=mean_harian[['titik', 'waktu', '_tanggal_dt', 'cumavg', 'mean_harian']],
        left_on='_tanggal_dt',
        right_on='_tanggal_dt',
        by=['titik', 'waktu'],
        direction='backward',
        allow_exact_matches=True
    )
    # Nilai awal: ambil cumavg dari hasil merge_asof
    merged[target_col] = merged['cumavg']

    # ==============================
    # 4) Isi baris yang lebih awal dari minimal tanggal B (per grup)
    #    dengan 1-hari mean dari satu tanggal acak per (titik, waktu)
    # ==============================
    # Cari minimal tanggal B per grup (titik, waktu)
    min_b_date = (
        mean_harian
        .groupby(['titik', 'waktu'], as_index=False)['_tanggal_dt']
        .min()
        .rename(columns={'_tanggal_dt': 'min_b_dt'})
    )
    merged = merged.merge(min_b_date, on=['titik', 'waktu'], how='left')

    # Tentukan baris yang perlu diisi dengan tanggal acak
    need_random = (
        merged['_tanggal_dt'].notna() &
        merged['min_b_dt'].notna() &
        (merged['_tanggal_dt'] < merged['min_b_dt'])
    )

    # Siapkan satu tanggal acak per grup dari domain mean_harian
    rng = np.random.default_rng(seed)
    random_date_map = {}
    for (t, w), sub in mean_harian.groupby(['titik', 'waktu']):
        dates = sub['_tanggal_dt'].unique()
        if len(dates) > 0:
            random_date_map[(t, w)] = rng.choice(dates)
        else:
            random_date_map[(t, w)] = pd.NaT

    # Ambil mean_harian pada tanggal acak itu per grup
    random_rows = []
    for (t, w), dt_choice in random_date_map.items():
        if pd.isna(dt_choice):
            continue
        val = mean_harian.loc[
            (mean_harian['titik'] == t) &
            (mean_harian['waktu'] == w) &
            (mean_harian['_tanggal_dt'] == dt_choice), 'mean_harian'
        ]
        if not val.empty:
            random_rows.append({'titik': t, 'waktu': w, 'random_mean_harian': float(val.iloc[0])})

    if random_rows:
        random_ref = pd.DataFrame(random_rows)
        merged = merged.merge(random_ref, on=['titik', 'waktu'], how='left')

        # Isi baris yang perlu tanggal acak dengan nilai tersebut
        idx = need_random[need_random].index
        merged.loc[idx, target_col] = merged.loc[idx, 'random_mean_harian']

    # ==============================
    # 5) Cleanup
    # ==============================
    # Buang kolom sementara
    drop_cols = ['_tanggal_dt', 'cumavg', 'mean_harian', 'min_b_dt', 'random_mean_harian']
    merged = merged.drop(columns=[c for c in drop_cols if c in merged.columns])

    return merged

# ---- contoh penggunaan ----
# df_fix_filled = fill_fix_with_b(df_fix, df_b, seed=123)
# df_fix_filled['jumlah burung pada titik x'].head()


In [64]:
# Cek tipe data kolom tanggal
print(fix['tanggal'].dtype)

# Jika tanggal bukan datetime, konversikan
fix['tanggal'] = pd.to_datetime(fix['tanggal'], errors='coerce')

# Hapus timezone jika ada
fix['tanggal'] = fix['tanggal'].dt.tz_localize(None)  # Hapus timezone jika ada

# Pastikan kolom jam dalam format string
fix['jam'] = fix['jam'].astype(str)

datetime64[ns]


In [65]:
fix.head()

,tanggal,jam,waktu,cuaca,jumlah burung pada titik x,titik,fase,strike
0,2006-01-25,1970-01-01T18:17:00.000Z,Sore,Berawan,NaN,1.0,Landing,1
1,2009-02-25,1970-01-01T16:10:00.000Z,Sore,Berawan,NaN,8.0,Landing,1
2,2021-02-25,1970-01-01T10:18:00.000Z,Siang,Hujan Gerimis,NaN,2.0,Take Off,1
3,2001-01-25,1970-01-01T06:33:00.000Z,Pagi,Badai Petir,NaN,3.0,Landing,1
4,2025-02-25,1970-01-01T12:55:00.000Z,Siang,Hujan Gerimis,NaN,7.0,Landing,1


In [66]:
# Cek tipe data kolom tanggal di df_b
print(b['tanggal'].dtype)

# Jika kolom 'tanggal' bukan datetime, konversi menjadi datetime
b['tanggal'] = pd.to_datetime(b['tanggal'], errors='coerce')

# Ubah kolom 'tanggal' ke format yymmdd
b['tanggal'] = b['tanggal'].dt.strftime('%y%m%d')  # format yymmdd

# Pastikan kolom 'jam' dalam format string
b['jam'] = b['jam'].astype(str)

# Cek hasil
print(b[['tanggal', 'jam']].head())


object
  tanggal         jam
0  150725  08:26:00.0
1  150725  08:26:00.0
2  150725  08:26:00.0
3  150725  08:26:00.0
4  150725  08:26:00.0


C:\Users\nalin\AppData\Local\Temp\ipykernel_28844\3495648250.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  b['tanggal'] = pd.to_datetime(b['tanggal'], errors='coerce')


In [67]:
b.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 643 entries, 0 to 642
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   latitude       643 non-null    float64
 1   longitude      643 non-null    object 
 2   lokasi         643 non-null    object 
 3   titik          643 non-null    int64  
 4   tanggal        643 non-null    object 
 5   jam            643 non-null    object 
 6   waktu          643 non-null    object 
 7   cuaca          643 non-null    object 
 8   jenis_burung   643 non-null    object 
 9   nama_ilmiah    643 non-null    object 
 10  jumlah_burung  643 non-null    int64  
 11  keterangan     20 non-null     object 
 12  dokumentasi    0 non-null      float64
 13  createdAt      643 non-null    object 
 14  updatedAt      0 non-null      float64
 15  deletedAt      0 non-null      float64
dtypes: float64(4), int64(2), object(10)
memory usage: 80.5+ KB


In [68]:
df_fix_filled = fill_fix_with_b(fix, b, seed=123)

C:\Users\nalin\AppData\Local\Temp\ipykernel_28844\2303418340.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_b['tanggal'] = pd.to_datetime(df_b['tanggal'], errors='coerce')


ValueError: left keys must be sorted

In [ ]:
df_fix_filled

#### <b> modeling

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)

# --- 1. Preprocessing ---
df_model = df.copy()

# Ekstrak fitur dari datetime
df_model["dayofweek"] = df_model["tanggal_fix"].dt.dayofweek
df_model["is_weekend"] = df_model["dayofweek"].isin([5,6]).astype(int)
df_model["hour"] = pd.to_datetime(df_model["jam"], errors="coerce").dt.hour

# Drop kolom yang redundant
X = df_model.drop(columns=["strike", "tanggal_fix", "jam", "tahun", "tanggal", "bulan"])
y = df_model["strike"]

# One-hot encoding untuk kategorikal
X = pd.get_dummies(X, columns=["waktu", "cuaca", "fase"], drop_first=True)

# --- 2. Train-test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# --- 3. Model training ---
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced_subsample",
    n_jobs=-1
)
model.fit(X_train, y_train)

# --- 4. Evaluasi dasar ---
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]  # probabilitas kelas 1

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, digits=4))

print("\n=== Confusion Matrix ===")
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[0,1], yticklabels=[0,1])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# --- 5. ROC Curve & AUC ---
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color="blue", label=f"ROC Curve (AUC = {roc_auc:.4f})")
plt.plot([0,1], [0,1], color="red", linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()

# --- 6. Precision-Recall Curve ---
precision, recall, _ = precision_recall_curve(y_test, y_proba)
avg_prec = average_precision_score(y_test, y_proba)

plt.figure(figsize=(6,6))
plt.plot(recall, precision, color="green", label=f"AP = {avg_prec:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend(loc="upper right")
plt.show()

# --- 7. Feature Importance ---
importances = model.feature_importances_
feat_names = X.columns
feat_importance = pd.DataFrame({"feature": feat_names, "importance": importances})
feat_importance = feat_importance.sort_values(by="importance", ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x="importance", y="feature", data=feat_importance.head(15))
plt.title("Top 15 Feature Importance (Random Forest)")
plt.show()

In [ ]:
# CatBoostClassifier
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# === 1. Pilih fitur & target ===
features = ["jumlah burung pada titik x", "titik", "hour", "dayofweek", "is_weekend", "waktu_Dini Hari", "waktu_Malam", "waktu_Pagi", "waktu_Siang", "waktu_Sore", "cuaca_Cerah Berawan", "cuaca_Hujan", "cuaca_Mendung", "fase_Take Off"]
target = "strike"

# Use the already processed X and y
# X = df_model[features]
# y = df_model[target]

# === 2. Train-test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# === 3. Identifikasi kolom kategori ===
# Since we used one-hot encoding, there are no categorical features in X_train
# If you want to use CatBoost's internal categorical handling, you would need to adjust preprocessing.
# For now, we will treat all features as numerical as they are already one-hot encoded or numerical.
cat_features = [] # Assuming X is already preprocessed with one-hot encoding

# === 4. Hitung scale_pos_weight ===
neg, pos = y_train.value_counts()
scale_pos_weight = neg / pos
print("scale_pos_weight:", scale_pos_weight)

# === 5. Definisikan CatBoost ===
cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    eval_metric="AUC",
    random_seed=42,
    verbose=200,
    # cat_features=cat_features, # Remove this if treating all as numerical after one-hot
    scale_pos_weight=scale_pos_weight
)

# === 6. Training ===
# Create CatBoost Pool if using internal categorical handling, otherwise fit directly
# train_pool = Pool(data=X_train, label=y_train, cat_features=cat_features)
# test_pool = Pool(data=X_test, label=y_test, cat_features=cat_features)

# cat_model.fit(train_pool, eval_set=test_pool, early_stopping_rounds=100)
cat_model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=100)


# === 7. Prediksi & Evaluasi ===
y_prob = cat_model.predict_proba(X_test)[:, 1]
y_pred = cat_model.predict(X_test)

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
# CatBoostClassifier with treshold tuning
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# ambil probabilitas kelas 1 dari CatBoost
y_prob = cat_model.predict_proba(X_test)[:, 1]

# coba beberapa threshold
thresholds = [0.5, 0.3, 0.2, 0.1] # dicoba semakin besar (0,9)

for thr in thresholds:
    print(f"\n===== Threshold: {thr} =====")
    y_pred_thr = (y_prob >= thr).astype(int)

    cm = confusion_matrix(y_test, y_pred_thr)
    print("Confusion Matrix:\n", cm)

    print(classification_report(y_test, y_pred_thr, digits=4))

    roc_auc = roc_auc_score(y_test, y_prob)
    print("ROC-AUC:", roc_auc)

In [ ]:
# EasyEnsembleClassifier (boosting khusus imbalance) & CatBoostClassifier (stabil di data imbalance)
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.ensemble import EasyEnsembleClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Model base
easy = EasyEnsembleClassifier(
    n_estimators=10,
    random_state=42
)

cat = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,  # pakai imbalance ratio
    verbose=0,
    random_state=42
)

# Meta-model (level-2)
meta = LogisticRegression(max_iter=1000, class_weight="balanced")

# Stacking Ensemble
stack_model = StackingClassifier(
    estimators=[('easy', easy), ('cat', cat)],
    final_estimator=meta,
    cv=5,
    n_jobs=-1,
    passthrough=True  # biar meta-model juga dapat input fitur asli
)

# Training
stack_model.fit(X_train, y_train)

# Evaluasi
y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

# Makin jelek jangan dipake
# DO NOT USE THIS